# 🤝 Collaborative Design Discussion

## Architectural Observations & Suggestions

After reviewing the SGS.ai codebase, here are my thoughts as a collaborator:

### ✅ What's Working Well

1. **Content-based identification (SHA1)** - Excellent choice for immutability and idempotency. This aligns perfectly with von Neumann's principles where identity is derived from structure.

2. **HLLSet algebra** - While inspired by HyperLogLog, HLLSets are fundamentally different. They compress entire datasets into **approximations of the original data** while preserving full set algebra operations (union, intersection, difference). Key properties:
   - Complies with all set operations
   - Recent research shows HLLSets comply with **F₂ ring** (Boolean rings)
   - Can be used to build **semantic lattices**
   - Memory efficient (KB for representing millions of elements)
   
   *Note: The F₂ ring and semantic lattice capabilities are not utilized in this project but represent future potential.*

3. **Edge versioning (head/tail pattern)** - The archival pattern for edges provides audit trails without losing history.

4. **Redis module stack** - Combining Redisearch + Roaring Bitmaps + Graph gives you a powerful in-memory foundation.

---

## 🎯 Design Rationale (Refined)

### Key Insight: Tokens Reference Atomic Sources Only

**Clarification from discussion**: Databases are just *one type* of source. Others include:
- Documents (PDFs, Word files)
- Emails
- User prompts
- APIs
- Etc.

**Therefore**: Token index should be **generic**, pointing to **leaf/atomic sources**:

| Source Domain | Atomic/Leaf Source | Compound Sources (via edges) |
|---------------|-------------------|------------------------------|
| Database | **Column** | Table → Database |
| File System | **File** | Folder → Drive |
| Email | **Email** | Mailbox → Account |
| User Prompt | **Prompt** | Session → User |

### Updated Token Index Design

```
┌─────────────────────────────────────────────────────────────┐
│  Token Index (Generic - Leaf References Only)               │
├─────────────────────────────────────────────────────────────┤
│  meta:tokens:{hash} → {                                     │
│      refs: "col_sha1,file_sha1,email_sha1,..."              │
│      source_types: "column,file,email,..."                  │
│      TF: 5                                                  │
│  }                                                          │
└─────────────────────────────────────────────────────────────┘
                           │
                           ▼ (traverse via edges to find compounds)
┌─────────────────────────────────────────────────────────────┐
│  Edge Index (Hierarchy - Updated on Ingestion)              │
├─────────────────────────────────────────────────────────────┤
│  column ──[contains_column]──▶ table                        │
│  table  ──[contains_table]───▶ db                           │
│  file   ──[in_folder]────────▶ folder                       │
│  email  ──[in_mailbox]───────▶ mailbox                      │
└─────────────────────────────────────────────────────────────┘
```

### Benefits of This Approach

1. **Generic & Extensible**: Same token index works for any data source type
2. **No Redundancy**: Compound sources are computed, not stored multiple times
3. **Edge-Driven Discovery**: Hierarchy traversal uses the same edge mechanism everywhere
4. **Consistent with von Neumann Model**: Edges *are* the structure

### Ingestion Workflow (Updated)

```python
# When ingesting tokens from ANY source:
ingest_tokens(r, tokens, source_sha1, source_type, [
    (parent1_sha1, parent1_type, edge_label1),
    (parent2_sha1, parent2_type, edge_label2),
    # ... chain up to root compound
])

# This automatically:
# 1. Updates token index with leaf ref
# 2. Creates edges from leaf → parent1 → parent2 → ...
```

---

## 🔄 The Query Flow (Updated)

```
User Prompt: "Show me Q4 2025 sales report"
                    │
                    ▼
┌─────────────────────────────────────────────────────────────┐
│  Step 1: Tokenize                                           │
│  tokens = ["q4", "2025", "sales", "report"]                 │
└─────────────────────────────────────────────────────────────┘
                    │
                    ▼
┌─────────────────────────────────────────────────────────────┐
│  Step 2: Token Lookup → Leaf Sources                        │
│  {                                                          │
│    "column": [col_revenue, col_quarter, ...],               │
│    "file": [file_report_pdf, ...],                          │
│    "email": [email_meeting, ...]                            │
│  }                                                          │
└─────────────────────────────────────────────────────────────┘
                    │
                    ▼
┌─────────────────────────────────────────────────────────────┐
│  Step 3: Resolve Compound Sources via Edges                 │
│  columns → tables → databases                               │
│  files → folders                                            │
│  emails → mailboxes                                         │
└─────────────────────────────────────────────────────────────┘
                    │
                    ▼
┌─────────────────────────────────────────────────────────────┐
│  Step 4: HLLSet Matching (Top-Down within each domain)      │
│  For DB domain: db_hll → table_hll → column_hll             │
│  For Files: folder_hll → file_hll                           │
└─────────────────────────────────────────────────────────────┘
                    │
                    ▼
┌─────────────────────────────────────────────────────────────┐
│  Step 5: Generate Domain-Specific Actions                   │
│  - SQL for databases                                        │
│  - File paths for documents                                 │
│  - Email retrieval for messages                             │
└─────────────────────────────────────────────────────────────┘
```

---

## 🔄 Separation of Concerns: meta_redis.py vs core_server.py

| Module | Responsibility | Transformer Alignment |
|--------|---------------|----------------------|
| `meta_redis.py` | Operational library (CRUD, indices, HLL ops, edge management) | **A** (Constructor), **B** (Copier) |
| `core_server.py` | Self-generative loop orchestration, prompt handling | **C** (Controller) |
| *Future*: `perceptron.py`? | Environment interface (different source adapters) | **D** (Perceptron) |

---

*Ready to test the updated token ingestion with generic leaf references?*

# SGS.ai Redis Development Notebook

This notebook is used for prototyping additional Redis functionality for SGS.ai:

1. **New `_create_index` functions** for DB, table, and columns
2. **Column loading methods** with HLLSet creation
3. **Redisearch index management** for DB, tables, columns, and edges

## Architecture Overview

```
┌─────────────────────────────────────────────────────────────┐
│                    Redis Memory Store                       │
├─────────────────────────────────────────────────────────────┤
│  Redisearch Indices:                                        │
│    - idx:db       (database metadata)                       │
│    - idx:table    (table metadata)                          │
│    - idx:column   (column metadata + HLLSet refs)           │
│    - edge:head    (current edges)                           │
│    - edge:tail    (archived edges)                          │
├─────────────────────────────────────────────────────────────┤
│  HLLSets (Roaring Bitmaps):                                 │
│    - rbs:{column_sha1}  (column cardinality estimation)     │
└─────────────────────────────────────────────────────────────┘
```

## Setup and Imports

In [ ]:
import sys
sys.path.insert(0, './sgs_core')

import mmh3
import json
import time
import uuid
import redis
import numpy as np
from typing import Dict, List, Optional, Tuple, Union
from redis.commands.search.field import TextField, NumericField, TagField
from redis.commands.search.indexDefinition import IndexDefinition, IndexType
from redis.commands.search.query import Query

# Import existing modules
from meta_algebra import HllSet
from meta_redis import RedisStore

print("Imports successful!")

## Redis Connection

In [ ]:
# Initialize Redis connection
# Use 'localhost' for local development or 'redis' for Docker
REDIS_HOST = 'localhost'  # Change to 'redis' if running in Docker
REDIS_PORT = 6379

try:
    r = redis.Redis(host=REDIS_HOST, port=REDIS_PORT, db=0, decode_responses=False)
    r.ping()
    print(f"✓ Connected to Redis at {REDIS_HOST}:{REDIS_PORT}")
    
    # Check loaded modules
    modules = r.execute_command('MODULE', 'LIST')
    print("\nLoaded modules:")
    for mod in modules:
        print(f"  - {mod}")
except redis.ConnectionError as e:
    print(f"✗ Failed to connect to Redis: {e}")

---
## Part 1: New Index Creation Functions

We need to create Redisearch indices for:
- **Database (DB)** metadata
- **Table** metadata
- **Column** metadata with HLLSet references

In [ ]:
def _create_db_index(r: redis.Redis, index_name: str = "idx:db") -> bool:
    """
    Create Redisearch index for database metadata.
    
    Schema:
        - db_sha1: Unique identifier (content-based hash)
        - db_name: Database name
        - db_type: Database type (e.g., 'postgresql', 'mysql', 'sqlite')
        - host: Database host
        - port: Database port
        - created_at: Timestamp of creation
        - updated_at: Timestamp of last update
        - tables: Comma-separated list of table SHA1s (TagField)
    
    Key pattern: meta:db:{db_sha1}
    """
    try:
        # Drop existing index if it exists
        try:
            r.ft(index_name).dropindex(delete_documents=False)
            print(f"Dropped existing index: {index_name}")
        except redis.exceptions.ResponseError:
            pass  # Index doesn't exist
        
        # Create index with schema
        r.ft(index_name).create_index(
            [
                TextField("db_sha1", sortable=True),
                TextField("db_name", sortable=True),
                TextField("db_type", sortable=True),
                TextField("host"),
                NumericField("port", sortable=True),
                NumericField("created_at", sortable=True),
                NumericField("updated_at", sortable=True),
                TagField("tables", separator=","),
            ],
            definition=IndexDefinition(prefix=["meta:db:"])
        )
        print(f"✓ Created index: {index_name}")
        return True
    except redis.exceptions.ResponseError as e:
        print(f"✗ Error creating index {index_name}: {e}")
        return False

# Test the function
_create_db_index(r)

In [ ]:
def _create_table_index(r: redis.Redis, index_name: str = "idx:table") -> bool:
    """
    Create Redisearch index for table metadata.
    
    Schema:
        - table_sha1: Unique identifier (content-based hash)
        - table_name: Table name
        - db_sha1: Reference to parent database
        - schema_name: Schema name (if applicable)
        - row_count: Estimated row count
        - created_at: Timestamp of creation
        - updated_at: Timestamp of last update
        - columns: Comma-separated list of column SHA1s (TagField)
    
    Key pattern: meta:table:{table_sha1}
    """
    try:
        try:
            r.ft(index_name).dropindex(delete_documents=False)
            print(f"Dropped existing index: {index_name}")
        except redis.exceptions.ResponseError:
            pass
        
        r.ft(index_name).create_index(
            [
                TextField("table_sha1", sortable=True),
                TextField("table_name", sortable=True),
                TextField("db_sha1", sortable=True),
                TextField("schema_name", sortable=True),
                NumericField("row_count", sortable=True),
                NumericField("created_at", sortable=True),
                NumericField("updated_at", sortable=True),
                TagField("columns", separator=","),
            ],
            definition=IndexDefinition(prefix=["meta:table:"])
        )
        print(f"✓ Created index: {index_name}")
        return True
    except redis.exceptions.ResponseError as e:
        print(f"✗ Error creating index {index_name}: {e}")
        return False

# Test the function
_create_table_index(r)

In [ ]:
def _create_column_index(r: redis.Redis, index_name: str = "idx:column") -> bool:
    """
    Create Redisearch index for column metadata with HLLSet references.
    
    Schema:
        - column_sha1: Unique identifier (content-based hash)
        - column_name: Column name
        - table_sha1: Reference to parent table
        - db_sha1: Reference to parent database
        - data_type: Column data type
        - hll_key: Key to the HLLSet storing column values
        - cardinality: Estimated unique value count from HLL
        - nullable: Whether column allows NULL values
        - created_at: Timestamp of creation
        - updated_at: Timestamp of last update
    
    Key pattern: meta:column:{column_sha1}
    """
    try:
        try:
            r.ft(index_name).dropindex(delete_documents=False)
            print(f"Dropped existing index: {index_name}")
        except redis.exceptions.ResponseError:
            pass
        
        r.ft(index_name).create_index(
            [
                TextField("column_sha1", sortable=True),
                TextField("column_name", sortable=True),
                TextField("table_sha1", sortable=True),
                TextField("db_sha1", sortable=True),
                TextField("data_type", sortable=True),
                TextField("hll_key"),
                NumericField("cardinality", sortable=True),
                NumericField("nullable", sortable=True),  # 0 or 1
                NumericField("created_at", sortable=True),
                NumericField("updated_at", sortable=True),
            ],
            definition=IndexDefinition(prefix=["meta:column:"])
        )
        print(f"✓ Created index: {index_name}")
        return True
    except redis.exceptions.ResponseError as e:
        print(f"✗ Error creating index {index_name}: {e}")
        return False

# Test the function
_create_column_index(r)

In [ ]:
def initialize_all_indices(r: redis.Redis) -> Dict[str, bool]:
    """
    Initialize all Redisearch indices for SGS.ai metadata management.
    
    Returns:
        Dictionary mapping index names to creation success status
    """
    results = {}
    results['idx:db'] = _create_db_index(r)
    results['idx:table'] = _create_table_index(r)
    results['idx:column'] = _create_column_index(r)
    
    print("\n" + "="*50)
    print("Index Initialization Summary:")
    for idx, success in results.items():
        status = "✓" if success else "✗"
        print(f"  {status} {idx}")
    
    return results

# Initialize all indices
initialize_all_indices(r)

---
## Part 1.5: Generic Token Index Design

### Design Clarification

Tokens should reference **atomic/leaf sources** only (the closest data container):
- In DB → **column** (not table, not database)
- In files → **file itself**
- In emails → **email**
- In prompts → **prompt**

**Compound sources** (DB, Table, Folder) are **inferred from edges**, not stored in token refs.

```
┌─────────────────────────────────────────────────────────────┐
│  Token Index (Leaf References Only)                         │
├─────────────────────────────────────────────────────────────┤
│  meta:tokens:{hash} → {                                     │
│      refs: "col_sha1,file_sha1,email_sha1,..."              │
│      source_types: "column,file,email,..."  (optional)      │
│      TF: 5                                                  │
│  }                                                          │
└─────────────────────────────────────────────────────────────┘
                           │
                           ▼ (traverse via edges)
┌─────────────────────────────────────────────────────────────┐
│  Edge Index (Hierarchy)                                     │
├─────────────────────────────────────────────────────────────┤
│  edge:head:{id} → column ──contains_column──▶ table        │
│  edge:head:{id} → table  ──contains_table───▶ db           │
│  edge:head:{id} → file   ──in_folder────────▶ folder       │
│  edge:head:{id} → email  ──in_mailbox───────▶ mailbox      │
└─────────────────────────────────────────────────────────────┘
```

### Ingestion Workflow

When ingesting tokens from a new source:
1. **Update token index** with leaf source ref
2. **Update/create edges** from leaf to compound parents
3. **Update compound source metadata** (optional, for caching)

In [ ]:
def _create_tokens_index(r: redis.Redis, index_name: str = "idx:tokens") -> bool:
    """
    Create generic tokens index with leaf-level references.
    
    Design principle: Tokens point to ATOMIC sources only (columns, files, emails).
    Compound sources (DB, Table, Folder) are inferred via edge traversal.
    
    Schema:
        - token_hash: MurmurHash3 of the token (20-digit padded)
        - token_text: Original token text (for debugging/display)
        - bin: HLL bin assignment (token_hash >> (64-P))
        - zeros: Leading zeros count for HLL
        - TF: Term frequency across all sources
        - refs: Comma-separated SHA1s of LEAF sources (columns, files, emails, etc.)
        - source_types: Comma-separated types matching refs order (column, file, email, etc.)
    
    Key pattern: meta:tokens:{token_hash:020}
    """
    try:
        try:
            r.ft(index_name).dropindex(delete_documents=False)
            print(f"Dropped existing index: {index_name}")
        except redis.exceptions.ResponseError:
            pass
        
        r.ft(index_name).create_index(
            [
                TextField("token_hash", sortable=True),
                TextField("token_text"),
                NumericField("bin", sortable=True),
                NumericField("zeros", sortable=True),
                NumericField("TF", sortable=True),
                TagField("refs", separator=","),          # Leaf source SHA1s
                TagField("source_types", separator=","),  # Corresponding types
            ],
            definition=IndexDefinition(prefix=["meta:tokens:"])
        )
        print(f"✓ Created generic token index: {index_name}")
        return True
    except redis.exceptions.ResponseError as e:
        print(f"✗ Error creating index {index_name}: {e}")
        return False

# Test generic token index
_create_tokens_index(r)

In [ ]:
def ingest_tokens(r: redis.Redis, tokens: List[str], 
                  source_sha1: str, source_type: str,
                  parent_chain: List[Tuple[str, str, str]] = None,
                  P: int = 10):
    """
    Ingest tokens from a leaf source with automatic edge management.
    
    This is the unified ingestion function that:
    1. Updates token index with leaf source reference
    2. Creates/updates edges from leaf source to compound parents
    
    Args:
        r: Redis connection
        tokens: List of tokens to index
        source_sha1: SHA1 of the LEAF source (column, file, email, etc.)
        source_type: Type of source ('column', 'file', 'email', 'prompt', etc.)
        parent_chain: List of (parent_sha1, parent_type, edge_label) tuples
                      describing the hierarchy from leaf to root.
                      Example for column: [
                          (table_sha1, 'table', 'contains_column'),
                          (db_sha1, 'db', 'contains_table')
                      ]
        P: HLL precision for bin calculation
    
    Example:
        # Ingesting tokens from a database column
        ingest_tokens(r, tokens, col_sha1, 'column', [
            (table_sha1, 'table', 'contains_column'),
            (db_sha1, 'db', 'contains_table')
        ])
        
        # Ingesting tokens from an email
        ingest_tokens(r, tokens, email_sha1, 'email', [
            (mailbox_sha1, 'mailbox', 'in_mailbox'),
            (account_sha1, 'account', 'owns_mailbox')
        ])
    """
    parent_chain = parent_chain or []
    timestamp = int(time.time() * 1000)
    
    # --- Step 1: Update Token Index ---
    pipe = r.pipeline()
    
    for token in tokens:
        # Calculate token hash
        token_hash, _ = mmh3.hash64(token)
        token_hash = token_hash & 0xFFFFFFFFFFFFFFFF  # Unsigned 64-bit
        token_key = f"meta:tokens:{token_hash:020}"
        
        # Get existing refs and types
        existing_refs = r.hget(token_key, "refs")
        existing_types = r.hget(token_key, "source_types")
        
        if existing_refs:
            refs_list = existing_refs.decode().split(",")
            types_list = existing_types.decode().split(",") if existing_types else []
        else:
            refs_list = []
            types_list = []
        
        # Add new ref if not already present
        if source_sha1 not in refs_list:
            refs_list.append(source_sha1)
            types_list.append(source_type)
        
        # Clean up empty strings
        refs_list = [r for r in refs_list if r]
        types_list = [t for t in types_list if t]
        
        # Update token metadata
        pipe.hincrby(token_key, "TF", 1)
        pipe.hset(token_key, "refs", ",".join(refs_list))
        pipe.hset(token_key, "source_types", ",".join(types_list))
        pipe.hsetnx(token_key, "token_hash", f"{token_hash:020}")
        pipe.hsetnx(token_key, "token_text", token[:100])  # Truncate long tokens
        pipe.hsetnx(token_key, "bin", token_hash >> (64 - P))
        zeros = (token_hash & -token_hash).bit_length() - 1 if token_hash != 0 else 0
        pipe.hsetnx(token_key, "zeros", zeros)
    
    pipe.execute()
    print(f"✓ Indexed {len(tokens)} tokens for {source_type}: {source_sha1[:12]}...")
    
    # --- Step 2: Update Edge Index for Parent Chain ---
    if parent_chain:
        current_sha1 = source_sha1
        current_type = source_type
        
        for parent_sha1, parent_type, edge_label in parent_chain:
            # Check if edge already exists
            edge_exists = _edge_exists(r, current_sha1, parent_sha1, edge_label)
            
            if not edge_exists:
                # Create new edge
                _create_edge_internal(r, current_sha1, parent_sha1, edge_label, {
                    "child_type": current_type,
                    "parent_type": parent_type,
                    "created_at": timestamp
                })
                print(f"  ✓ Created edge: {current_type} --[{edge_label}]--> {parent_type}")
            
            # Move up the chain
            current_sha1 = parent_sha1
            current_type = parent_type

def _edge_exists(r: redis.Redis, left_sha1: str, right_sha1: str, label: str) -> bool:
    """Check if an edge already exists between two entities."""
    try:
        # Search for existing edge
        query = Query(f"@left:{left_sha1} @right:{right_sha1} @label:{label}")
        results = r.ft("edge:head").search(query)
        return results.total > 0
    except:
        return False

def _create_edge_internal(r: redis.Redis, left_sha1: str, right_sha1: str,
                          label: str, metadata: dict) -> str:
    """Internal edge creation without printing."""
    commit_id = str(uuid.uuid1())
    timestamp = int(time.time() * 1000)
    
    attr = json.dumps(metadata or {})
    edge_data = {
        "label": label,
        "left": left_sha1,
        "right": right_sha1,
        "attr": attr
    }
    
    edge_sha1 = generate_content_hash(edge_data)
    edge_key = f"edge:head:{commit_id}:{edge_sha1}"
    edge_data["e_sha1"] = edge_sha1
    edge_data["timestamp"] = timestamp
    
    r.hset(edge_key, mapping=edge_data)
    return edge_sha1

# Test: Ingest tokens from different source types
print("="*60)
print("Testing Generic Token Ingestion")
print("="*60)

# Simulate a database column
col_sha1_test = "col_sha1_revenue_12345"
table_sha1_test = "table_sha1_sales_67890"
db_sha1_test = "db_sha1_analytics_abcde"

column_tokens = ["revenue", "sales", "amount", "total", "q4", "2025"]
ingest_tokens(r, column_tokens, col_sha1_test, "column", [
    (table_sha1_test, "table", "contains_column"),
    (db_sha1_test, "db", "contains_table")
])

# Simulate an email
email_sha1_test = "email_sha1_meeting_xyz"
mailbox_sha1_test = "mailbox_sha1_inbox_abc"

email_tokens = ["meeting", "sales", "quarterly", "review", "budget"]
ingest_tokens(r, email_tokens, email_sha1_test, "email", [
    (mailbox_sha1_test, "mailbox", "in_mailbox")
])

# Simulate a document/file
file_sha1_test = "file_sha1_report_pdf"
folder_sha1_test = "folder_sha1_reports"

file_tokens = ["quarterly", "sales", "report", "2025", "analysis"]
ingest_tokens(r, file_tokens, file_sha1_test, "file", [
    (folder_sha1_test, "folder", "in_folder")
])

# Verify: check the "sales" token which appears in all sources
print("\n" + "-"*40)
print("Verifying 'sales' token (appears in column, email, and file):")
sales_hash, _ = mmh3.hash64("sales")
sales_hash = sales_hash & 0xFFFFFFFFFFFFFFFF
sales_key = f"meta:tokens:{sales_hash:020}"
token_data = r.hgetall(sales_key)
for k, v in token_data.items():
    print(f"  {k.decode()}: {v.decode()}")

---
## Part 2: Column Loading and HLLSet Creation

Methods to load columns from tables and create HLLSets for cardinality estimation.

In [ ]:
def generate_content_hash(data: dict) -> str:
    """
    Generate a content-based SHA1 hash for identification.
    Uses HllSet for consistent hashing across the system.
    """
    hll = HllSet()
    hll.from_dict(data)
    return hll.id()

def register_database(r: redis.Redis, db_name: str, db_type: str, 
                      host: str = "localhost", port: int = 5432) -> str:
    """
    Register a database in the metadata store.
    
    Args:
        r: Redis connection
        db_name: Name of the database
        db_type: Type of database (postgresql, mysql, etc.)
        host: Database host
        port: Database port
    
    Returns:
        db_sha1: Content-based hash identifier for the database
    """
    timestamp = int(time.time() * 1000)
    
    # Generate content-based hash
    db_data = {
        "db_name": db_name,
        "db_type": db_type,
        "host": host,
        "port": port
    }
    db_sha1 = generate_content_hash(db_data)
    
    # Store in Redis
    key = f"meta:db:{db_sha1}"
    r.hset(key, mapping={
        "db_sha1": db_sha1,
        "db_name": db_name,
        "db_type": db_type,
        "host": host,
        "port": port,
        "created_at": timestamp,
        "updated_at": timestamp,
        "tables": ""
    })
    
    print(f"✓ Registered database: {db_name} (sha1: {db_sha1[:12]}...)")
    return db_sha1

# Test: Register a sample database
test_db_sha1 = register_database(r, "test_db", "postgresql", "localhost", 5432)
print(f"Database SHA1: {test_db_sha1}")

In [ ]:
def register_table(r: redis.Redis, table_name: str, db_sha1: str,
                   schema_name: str = "public", row_count: int = 0) -> str:
    """
    Register a table in the metadata store and update parent database.
    
    Args:
        r: Redis connection
        table_name: Name of the table
        db_sha1: Parent database SHA1
        schema_name: Schema name (default 'public')
        row_count: Estimated row count
    
    Returns:
        table_sha1: Content-based hash identifier for the table
    """
    timestamp = int(time.time() * 1000)
    
    # Generate content-based hash
    table_data = {
        "table_name": table_name,
        "db_sha1": db_sha1,
        "schema_name": schema_name
    }
    table_sha1 = generate_content_hash(table_data)
    
    # Store table metadata
    key = f"meta:table:{table_sha1}"
    r.hset(key, mapping={
        "table_sha1": table_sha1,
        "table_name": table_name,
        "db_sha1": db_sha1,
        "schema_name": schema_name,
        "row_count": row_count,
        "created_at": timestamp,
        "updated_at": timestamp,
        "columns": ""
    })
    
    # Update parent database's tables list
    db_key = f"meta:db:{db_sha1}"
    existing_tables = r.hget(db_key, "tables")
    if existing_tables:
        tables_set = set(existing_tables.decode().split(",")) if existing_tables else set()
    else:
        tables_set = set()
    tables_set.discard("")  # Remove empty string if present
    tables_set.add(table_sha1)
    r.hset(db_key, "tables", ",".join(tables_set))
    r.hset(db_key, "updated_at", timestamp)
    
    print(f"✓ Registered table: {table_name} (sha1: {table_sha1[:12]}...)")
    return table_sha1

# Test: Register a sample table
test_table_sha1 = register_table(r, "users", test_db_sha1, "public", 10000)
print(f"Table SHA1: {test_table_sha1}")

In [ ]:
def load_column_with_hllset(r: redis.Redis, column_name: str, table_sha1: str,
                            db_sha1: str, column_values: List[str],
                            data_type: str = "varchar", nullable: bool = True,
                            P: int = 10) -> Tuple[str, HllSet]:
    """
    Load a column and create HLLSet for cardinality estimation.
    
    This method:
    1. Creates an HLLSet from column values
    2. Stores the HLLSet in Redis (Roaring Bitmap)
    3. Updates column metadata with HLL reference
    4. Updates parent table's columns list
    
    Args:
        r: Redis connection
        column_name: Name of the column
        table_sha1: Parent table SHA1
        db_sha1: Parent database SHA1
        column_values: List of values in the column
        data_type: Column data type
        nullable: Whether column allows NULLs
        P: HLL precision (default 10 = 1024 registers)
    
    Returns:
        Tuple of (column_sha1, HllSet)
    """
    timestamp = int(time.time() * 1000)
    
    # Create HLLSet from column values
    hll = HllSet(P)
    for value in column_values:
        if value is not None:  # Skip NULL values
            hll.add(str(value))
    
    # Generate content-based hash for column
    column_data = {
        "column_name": column_name,
        "table_sha1": table_sha1,
        "db_sha1": db_sha1,
        "data_type": data_type
    }
    column_sha1 = generate_content_hash(column_data)
    
    # Store HLLSet in Redis using Roaring Bitmap
    hll_key = f"rbs:col:{column_sha1}"
    counts = hll.counts.flatten()
    pipe = r.pipeline()
    for index, value in enumerate(counts):
        if value:  # Only set bits for non-zero values
            pipe.execute_command("SETBIT", hll_key, index, 1)
    pipe.execute()
    
    # Estimate cardinality
    cardinality = hll.count()
    
    # Store column metadata
    key = f"meta:column:{column_sha1}"
    r.hset(key, mapping={
        "column_sha1": column_sha1,
        "column_name": column_name,
        "table_sha1": table_sha1,
        "db_sha1": db_sha1,
        "data_type": data_type,
        "hll_key": hll_key,
        "cardinality": cardinality,
        "nullable": 1 if nullable else 0,
        "created_at": timestamp,
        "updated_at": timestamp
    })
    
    # Update parent table's columns list
    table_key = f"meta:table:{table_sha1}"
    existing_columns = r.hget(table_key, "columns")
    if existing_columns:
        columns_set = set(existing_columns.decode().split(",")) if existing_columns else set()
    else:
        columns_set = set()
    columns_set.discard("")  # Remove empty string if present
    columns_set.add(column_sha1)
    r.hset(table_key, "columns", ",".join(columns_set))
    r.hset(table_key, "updated_at", timestamp)
    
    print(f"✓ Loaded column: {column_name}")
    print(f"  - SHA1: {column_sha1[:12]}...")
    print(f"  - HLL Key: {hll_key}")
    print(f"  - Estimated cardinality: {cardinality}")
    print(f"  - Actual unique values: {len(set(v for v in column_values if v is not None))}")
    
    return column_sha1, hll

# Test: Load sample column data
sample_emails = [
    "user1@example.com", "user2@example.com", "user3@example.com",
    "user1@example.com",  # duplicate
    "user4@example.com", "user5@example.com", "user6@example.com",
    None,  # NULL value
    "user7@example.com", "user8@example.com"
]

email_sha1, email_hll = load_column_with_hllset(
    r, "email", test_table_sha1, test_db_sha1, 
    sample_emails, "varchar", True
)

In [ ]:
def load_multiple_columns(r: redis.Redis, table_sha1: str, db_sha1: str,
                          columns: Dict[str, Dict]) -> Dict[str, Tuple[str, HllSet]]:
    """
    Load multiple columns at once with their HLLSets.
    
    Args:
        r: Redis connection
        table_sha1: Parent table SHA1
        db_sha1: Parent database SHA1
        columns: Dictionary of column definitions:
            {
                "column_name": {
                    "values": [...],
                    "data_type": "varchar",
                    "nullable": True
                }
            }
    
    Returns:
        Dictionary mapping column names to (sha1, HllSet) tuples
    """
    results = {}
    
    for col_name, col_def in columns.items():
        values = col_def.get("values", [])
        data_type = col_def.get("data_type", "varchar")
        nullable = col_def.get("nullable", True)
        
        sha1, hll = load_column_with_hllset(
            r, col_name, table_sha1, db_sha1,
            values, data_type, nullable
        )
        results[col_name] = (sha1, hll)
    
    return results

# Test: Load multiple columns
columns_data = {
    "user_id": {
        "values": [str(i) for i in range(1, 101)],
        "data_type": "integer",
        "nullable": False
    },
    "username": {
        "values": [f"user_{i}" for i in range(1, 101)],
        "data_type": "varchar",
        "nullable": False
    },
    "status": {
        "values": ["active", "inactive", "pending"] * 33 + ["active"],
        "data_type": "varchar",
        "nullable": True
    }
}

loaded_columns = load_multiple_columns(r, test_table_sha1, test_db_sha1, columns_data)

---
## Part 2.5: Prompt-to-Data Hierarchical Matching

This is the core algorithm for matching user prompts to relevant data sources using:
1. Token index lookup for initial candidate discovery
2. HLLSet matching for hierarchical relevance scoring
3. Top-down traversal with pruning

In [ ]:
# Configuration for hierarchical matching
MATCH_THRESHOLDS = {
    "db": 0.2,      # Low threshold - cast wide net at DB level
    "table": 0.4,   # Medium - start narrowing
    "column": 0.6   # Higher - need good match for columns
}

def tokenize_prompt(prompt: str) -> List[str]:
    """
    Simple tokenization for user prompts.
    In production, consider using proper NLP tokenization.
    """
    import re
    # Lowercase, remove punctuation, split on whitespace
    tokens = re.findall(r'\b[a-zA-Z0-9]+\b', prompt.lower())
    # Remove common stop words
    stop_words = {'the', 'a', 'an', 'is', 'are', 'was', 'were', 'be', 'been', 
                  'being', 'have', 'has', 'had', 'do', 'does', 'did', 'will',
                  'would', 'could', 'should', 'may', 'might', 'must', 'shall',
                  'can', 'for', 'and', 'or', 'but', 'in', 'on', 'at', 'to',
                  'from', 'with', 'by', 'of', 'that', 'this', 'it', 'its',
                  'me', 'my', 'i', 'you', 'your', 'we', 'our', 'they', 'their'}
    return [t for t in tokens if t not in stop_words and len(t) > 1]

def create_prompt_hllset(tokens: List[str], P: int = 10) -> HllSet:
    """Create HLLSet from prompt tokens."""
    hll = HllSet(P)
    for token in tokens:
        hll.add(token)
    return hll

# Test tokenization
test_prompt = "Show me all sales data for Q4 2025 with revenue breakdown"
prompt_tokens = tokenize_prompt(test_prompt)
print(f"Original: {test_prompt}")
print(f"Tokens: {prompt_tokens}")

In [ ]:
def lookup_token_refs(r: redis.Redis, tokens: List[str]) -> Dict[str, List[str]]:
    """
    Look up token references from the index.
    
    Returns leaf source SHA1s grouped by source type.
    Compound sources (DB, Table, Folder) are resolved via edge traversal.
    
    Returns:
        Dictionary with structure:
        {
            'column': ['sha1_1', 'sha1_2', ...],
            'file': ['sha1_3', ...],
            'email': ['sha1_4', ...],
            ...
        }
    """
    refs_by_type = {}
    
    for token in tokens:
        token_hash, _ = mmh3.hash64(token)
        token_hash = token_hash & 0xFFFFFFFFFFFFFFFF
        token_key = f"meta:tokens:{token_hash:020}"
        
        token_data = r.hgetall(token_key)
        if token_data:
            refs_str = token_data.get(b"refs", b"").decode()
            types_str = token_data.get(b"source_types", b"").decode()
            
            refs = [r for r in refs_str.split(",") if r]
            types = [t for t in types_str.split(",") if t]
            
            # Pair refs with their types
            for ref, source_type in zip(refs, types):
                if source_type not in refs_by_type:
                    refs_by_type[source_type] = set()
                refs_by_type[source_type].add(ref)
    
    # Convert sets to lists
    return {k: list(v) for k, v in refs_by_type.items()}

def resolve_compound_sources(r: redis.Redis, leaf_refs: Dict[str, List[str]], 
                             target_type: str) -> List[str]:
    """
    Resolve compound sources by traversing edges from leaf sources.
    
    Example: Given column refs, find all parent databases.
    
    Args:
        r: Redis connection
        leaf_refs: Dictionary of leaf refs by type (from lookup_token_refs)
        target_type: The compound type to resolve to ('db', 'table', 'folder', etc.)
    
    Returns:
        List of unique SHA1s of the target compound type
    """
    compound_refs = set()
    
    # Define traversal paths: leaf_type -> [(edge_label, intermediate_type), ...]
    traversal_paths = {
        'column': {
            'table': [('contains_column', 'table')],
            'db': [('contains_column', 'table'), ('contains_table', 'db')]
        },
        'file': {
            'folder': [('in_folder', 'folder')]
        },
        'email': {
            'mailbox': [('in_mailbox', 'mailbox')],
            'account': [('in_mailbox', 'mailbox'), ('owns_mailbox', 'account')]
        }
    }
    
    for source_type, refs in leaf_refs.items():
        if source_type in traversal_paths and target_type in traversal_paths[source_type]:
            path = traversal_paths[source_type][target_type]
            
            for ref in refs:
                current_ref = ref
                for edge_label, next_type in path:
                    # Find edge from current to parent
                    try:
                        query = Query(f"@left:{current_ref} @label:{edge_label}")
                        results = r.ft("edge:head").search(query)
                        
                        if results.total > 0:
                            # Get the parent (right side of edge)
                            current_ref = getattr(results.docs[0], 'right', None)
                        else:
                            current_ref = None
                            break
                    except:
                        current_ref = None
                        break
                
                if current_ref:
                    compound_refs.add(current_ref)
    
    return list(compound_refs)

# Test token lookup with generic refs
print("="*60)
print("Testing Generic Token Lookup")
print("="*60)

prompt_tokens = ["sales", "quarterly", "2025"]
leaf_refs = lookup_token_refs(r, prompt_tokens)

print(f"Prompt tokens: {prompt_tokens}")
print(f"\nLeaf references by type:")
for source_type, refs in leaf_refs.items():
    print(f"  {source_type}: {len(refs)} refs")
    for ref in refs[:3]:  # Show first 3
        print(f"    - {ref[:20]}...")

# Resolve to compound sources
print(f"\nResolved compound sources:")
tables = resolve_compound_sources(r, leaf_refs, 'table')
print(f"  Tables: {len(tables)}")
dbs = resolve_compound_sources(r, leaf_refs, 'db')
print(f"  Databases: {len(dbs)}")

In [ ]:
from dataclasses import dataclass
from typing import NamedTuple

@dataclass
class MatchResult:
    """Result of hierarchical matching."""
    sha1: str
    level: str  # 'db', 'table', 'column'
    name: str
    score: float
    parent_sha1: Optional[str] = None
    children: List['MatchResult'] = None
    
    def __post_init__(self):
        if self.children is None:
            self.children = []

def hierarchical_match(r: redis.Redis, prompt: str, 
                       thresholds: Dict[str, float] = None) -> List[MatchResult]:
    """
    Main entry point for prompt-to-data matching.
    
    Algorithm:
    1. Tokenize prompt and create prompt HLLSet
    2. Look up candidate SHA1s from token index
    3. For each DB candidate:
       - Calculate match score with prompt HLLSet
       - If above threshold, traverse to tables
    4. For each Table candidate (under matching DBs):
       - Calculate match score
       - If above threshold, traverse to columns
    5. Return hierarchical match results
    
    Args:
        r: Redis connection
        prompt: User's natural language prompt
        thresholds: Match thresholds per level (defaults to MATCH_THRESHOLDS)
    
    Returns:
        List of MatchResult objects representing matching hierarchy
    """
    thresholds = thresholds or MATCH_THRESHOLDS
    
    # Step 1: Tokenize and create prompt HLLSet
    tokens = tokenize_prompt(prompt)
    if not tokens:
        print("Warning: No tokens extracted from prompt")
        return []
    
    prompt_hll = create_prompt_hllset(tokens)
    print(f"Prompt tokens: {tokens}")
    print(f"Prompt HLLSet cardinality: {prompt_hll.count()}")
    
    # Step 2: Get candidate refs from token index
    candidate_refs = lookup_token_refs(r, tokens)
    
    # Step 3: Match at DB level
    db_matches = []
    for db_sha1 in candidate_refs['db']:
        db_meta = r.hgetall(f"meta:db:{db_sha1}")
        if not db_meta:
            continue
        
        db_name = db_meta.get(b'db_name', b'').decode()
        
        # Get DB's HLLSet (would be from rbs:db:{sha1})
        # For now, create from DB metadata tokens
        db_tokens = [db_name] + db_meta.get(b'db_type', b'').decode().split()
        db_hll = create_prompt_hllset(db_tokens)
        
        # Calculate match score
        try:
            score = prompt_hll.match(db_hll) / 100.0  # Normalize to 0-1
        except:
            score = 0.0
        
        if score >= thresholds['db']:
            match_result = MatchResult(
                sha1=db_sha1,
                level='db',
                name=db_name,
                score=score
            )
            
            # Step 4: Traverse to tables
            tables_str = db_meta.get(b'tables', b'').decode()
            table_sha1s = [t for t in tables_str.split(',') if t]
            
            for table_sha1 in table_sha1s:
                # Only process if table is also in candidates OR is child of matching DB
                table_meta = r.hgetall(f"meta:table:{table_sha1}")
                if not table_meta:
                    continue
                
                table_name = table_meta.get(b'table_name', b'').decode()
                
                # Create table HLLSet from name + schema
                table_tokens = [table_name] + table_meta.get(b'schema_name', b'').decode().split()
                table_hll = create_prompt_hllset(table_tokens)
                
                try:
                    table_score = prompt_hll.match(table_hll) / 100.0
                except:
                    table_score = 0.0
                
                if table_score >= thresholds['table']:
                    table_match = MatchResult(
                        sha1=table_sha1,
                        level='table',
                        name=table_name,
                        score=table_score,
                        parent_sha1=db_sha1
                    )
                    
                    # Step 5: Traverse to columns
                    columns_str = table_meta.get(b'columns', b'').decode()
                    column_sha1s = [c for c in columns_str.split(',') if c]
                    
                    for col_sha1 in column_sha1s:
                        col_meta = r.hgetall(f"meta:column:{col_sha1}")
                        if not col_meta:
                            continue
                        
                        col_name = col_meta.get(b'column_name', b'').decode()
                        
                        # For columns, we'd ideally match against the column's value HLLSet
                        # For now, match against column name
                        col_tokens = [col_name] + col_meta.get(b'data_type', b'').decode().split()
                        col_hll = create_prompt_hllset(col_tokens)
                        
                        try:
                            col_score = prompt_hll.match(col_hll) / 100.0
                        except:
                            col_score = 0.0
                        
                        if col_score >= thresholds['column']:
                            col_match = MatchResult(
                                sha1=col_sha1,
                                level='column',
                                name=col_name,
                                score=col_score,
                                parent_sha1=table_sha1
                            )
                            table_match.children.append(col_match)
                    
                    match_result.children.append(table_match)
            
            db_matches.append(match_result)
    
    return db_matches

def print_match_tree(matches: List[MatchResult], indent: int = 0):
    """Pretty print the match hierarchy."""
    for match in matches:
        prefix = "  " * indent
        icon = {"db": "🗄️", "table": "📋", "column": "📊"}.get(match.level, "•")
        print(f"{prefix}{icon} {match.name} ({match.level}) - score: {match.score:.2%}")
        if match.children:
            print_match_tree(match.children, indent + 1)

# Test the hierarchical matching
print("="*60)
print("Testing Hierarchical Match")
print("="*60)
matches = hierarchical_match(r, test_prompt)
print("\nMatch Results:")
print_match_tree(matches)

In [ ]:
def generate_sql_from_matches(matches: List[MatchResult]) -> List[str]:
    """
    Generate candidate SQL statements from match results.
    
    This is a simplified SQL generator. In production, you'd want:
    - Proper schema awareness
    - Join detection from edges
    - Aggregation inference from prompt
    """
    sql_statements = []
    
    for db_match in matches:
        for table_match in db_match.children:
            if table_match.children:
                # We have matching columns
                columns = [col.name for col in table_match.children]
                columns_str = ", ".join(columns)
                
                sql = f"SELECT {columns_str} FROM {table_match.name}"
                sql_statements.append({
                    'sql': sql,
                    'db': db_match.name,
                    'table': table_match.name,
                    'columns': columns,
                    'confidence': sum(c.score for c in table_match.children) / len(table_match.children)
                })
            else:
                # No specific columns matched, select all
                sql = f"SELECT * FROM {table_match.name}"
                sql_statements.append({
                    'sql': sql,
                    'db': db_match.name,
                    'table': table_match.name,
                    'columns': ['*'],
                    'confidence': table_match.score
                })
    
    return sql_statements

# Test SQL generation
print("\nGenerated SQL Candidates:")
print("-" * 40)
sql_candidates = generate_sql_from_matches(matches)
for i, candidate in enumerate(sql_candidates, 1):
    print(f"{i}. {candidate['sql']}")
    print(f"   Database: {candidate['db']}, Confidence: {candidate['confidence']:.2%}")
    print()

---
## Part 2.6: End-to-End Test Scenario

Complete workflow demonstration: Register data → Index tokens → Process prompt → Generate SQL

In [ ]:
def run_e2e_scenario(r: redis.Redis):
    """
    Complete end-to-end demonstration of the SGS.ai data discovery workflow.
    Now using generic token ingestion with edge-based hierarchy.
    """
    print("="*70)
    print("  END-TO-END SCENARIO: Multi-Source Data Discovery")
    print("="*70)
    
    # Clean up any previous test data
    for pattern in ["meta:db:*", "meta:table:*", "meta:column:*", "meta:tokens:*", "rbs:*", "edge:*"]:
        keys = r.keys(pattern)
        if keys:
            r.delete(*keys)
    print("\n✓ Cleaned up previous test data")
    
    # Initialize indices
    print("\n--- Step 1: Initialize Indices ---")
    initialize_all_indices(r)
    _create_tokens_index(r)  # Generic token index
    
    # ============================================
    # SOURCE 1: Database (Sales Analytics)
    # ============================================
    print("\n--- Step 2a: Register Database Source ---")
    
    sales_db_sha1 = register_database(r, "sales_analytics", "postgresql", "db.company.com", 5432)
    sales_table_sha1 = register_table(r, "quarterly_sales", sales_db_sha1, "public", 50000)
    
    # Columns with realistic sales data
    sales_columns = {
        "quarter": {
            "values": ["Q1", "Q2", "Q3", "Q4"] * 12500,
            "data_type": "varchar",
            "nullable": False
        },
        "year": {
            "values": [str(y) for y in [2023, 2024, 2025] for _ in range(16667)],
            "data_type": "integer",
            "nullable": False
        },
        "revenue": {
            "values": [f"{i*1000}" for i in range(1, 50001)],
            "data_type": "decimal",
            "nullable": False
        },
        "region": {
            "values": ["North", "South", "East", "West"] * 12500,
            "data_type": "varchar",
            "nullable": True
        }
    }
    
    loaded = load_multiple_columns(r, sales_table_sha1, sales_db_sha1, sales_columns)
    
    # Ingest tokens from columns (using generic ingestion)
    print("\n--- Step 2b: Ingest Tokens from Columns ---")
    for col_name, (col_sha1, _) in loaded.items():
        col_tokens = [col_name.lower()] + col_name.lower().replace("_", " ").split()
        if col_name == "quarter":
            col_tokens.extend(["q1", "q2", "q3", "q4", "quarterly"])
        elif col_name == "year":
            col_tokens.extend(["2023", "2024", "2025"])
        elif col_name == "revenue":
            col_tokens.extend(["sales", "amount", "total", "money"])
        elif col_name == "region":
            col_tokens.extend(["north", "south", "east", "west", "regional"])
        
        # Ingest with parent chain (column → table → db)
        ingest_tokens(r, col_tokens, col_sha1, "column", [
            (sales_table_sha1, "table", "contains_column"),
            (sales_db_sha1, "db", "contains_table")
        ])
    
    # ============================================
    # SOURCE 2: Document (Report PDF)
    # ============================================
    print("\n--- Step 2c: Register Document Source ---")
    
    # Register folder and file
    reports_folder_sha1 = generate_content_hash({"name": "reports", "type": "folder"})
    r.hset(f"meta:folder:{reports_folder_sha1}", mapping={
        "folder_sha1": reports_folder_sha1,
        "folder_name": "reports",
        "created_at": int(time.time() * 1000)
    })
    
    report_file_sha1 = generate_content_hash({"name": "Q4_2025_Sales_Report.pdf", "folder": reports_folder_sha1})
    r.hset(f"meta:file:{report_file_sha1}", mapping={
        "file_sha1": report_file_sha1,
        "file_name": "Q4_2025_Sales_Report.pdf",
        "file_type": "pdf",
        "folder_sha1": reports_folder_sha1,
        "created_at": int(time.time() * 1000)
    })
    
    # Ingest tokens from file
    file_tokens = ["quarterly", "sales", "report", "q4", "2025", "revenue", 
                   "analysis", "performance", "growth", "regional"]
    ingest_tokens(r, file_tokens, report_file_sha1, "file", [
        (reports_folder_sha1, "folder", "in_folder")
    ])
    
    # ============================================
    # SOURCE 3: Email
    # ============================================
    print("\n--- Step 2d: Register Email Source ---")
    
    inbox_sha1 = generate_content_hash({"name": "inbox", "type": "mailbox"})
    r.hset(f"meta:mailbox:{inbox_sha1}", mapping={
        "mailbox_sha1": inbox_sha1,
        "mailbox_name": "inbox",
        "created_at": int(time.time() * 1000)
    })
    
    email_sha1 = generate_content_hash({"subject": "Q4 Sales Review Meeting", "mailbox": inbox_sha1})
    r.hset(f"meta:email:{email_sha1}", mapping={
        "email_sha1": email_sha1,
        "subject": "Q4 Sales Review Meeting",
        "mailbox_sha1": inbox_sha1,
        "created_at": int(time.time() * 1000)
    })
    
    # Ingest tokens from email
    email_tokens = ["meeting", "sales", "review", "q4", "quarterly", "budget",
                    "targets", "performance", "team"]
    ingest_tokens(r, email_tokens, email_sha1, "email", [
        (inbox_sha1, "mailbox", "in_mailbox")
    ])
    
    # ============================================
    # Step 3: Test Multi-Source Discovery
    # ============================================
    print("\n" + "="*70)
    print("  TESTING MULTI-SOURCE PROMPT MATCHING")
    print("="*70)
    
    test_prompts = [
        "Show me Q4 2025 sales revenue",
        "Find the quarterly sales report",
        "What meetings are about sales review?",
        "Regional performance data"
    ]
    
    for prompt in test_prompts:
        print(f"\n{'='*60}")
        print(f"PROMPT: \"{prompt}\"")
        print('='*60)
        
        tokens = tokenize_prompt(prompt)
        print(f"Tokens: {tokens}")
        
        # Get leaf references
        leaf_refs = lookup_token_refs(r, tokens)
        
        print(f"\nLeaf sources found:")
        for source_type, refs in leaf_refs.items():
            print(f"  📌 {source_type}: {len(refs)} matches")
            for ref in refs[:2]:  # Show first 2
                # Try to get name
                for prefix in ['meta:column:', 'meta:file:', 'meta:email:']:
                    data = r.hgetall(f"{prefix}{ref}")
                    if data:
                        name_key = [k for k in data.keys() if b'name' in k or b'subject' in k]
                        if name_key:
                            print(f"      → {data[name_key[0]].decode()}")
                        break
        
        # Show potential actions by source type
        print(f"\nPotential actions:")
        if 'column' in leaf_refs:
            # Resolve to tables
            tables = resolve_compound_sources(r, leaf_refs, 'table')
            if tables:
                print(f"  🗄️  Generate SQL query against {len(tables)} table(s)")
        if 'file' in leaf_refs:
            print(f"  📄 Retrieve {len(leaf_refs['file'])} document(s)")
        if 'email' in leaf_refs:
            print(f"  ✉️  Fetch {len(leaf_refs['email'])} email(s)")
    
    print("\n" + "="*70)
    print("  END-TO-END SCENARIO COMPLETE")
    print("="*70)
    
    # Summary statistics
    print("\n📊 Summary Statistics:")
    print(f"  Tokens indexed: {len(r.keys('meta:tokens:*'))}")
    print(f"  Edges created: {len(r.keys('edge:head:*'))}")
    print(f"  Columns: {len(r.keys('meta:column:*'))}")
    print(f"  Files: {len(r.keys('meta:file:*'))}")
    print(f"  Emails: {len(r.keys('meta:email:*'))}")

# Run the full scenario
run_e2e_scenario(r)

---
## Part 3: Edge Management for Column Relationships

Create and manage edges between database entities (DB → Table → Column).

In [ ]:
def create_entity_edge(r: redis.Redis, left_sha1: str, right_sha1: str,
                       label: str, metadata: Optional[Dict] = None) -> str:
    """
    Create an edge between two entities.
    
    Edge types:
        - "contains_table": DB → Table
        - "contains_column": Table → Column
        - "references": Column → Column (foreign key)
        - "similar_to": Column → Column (based on HLL similarity)
    
    Args:
        r: Redis connection
        left_sha1: Source entity SHA1
        right_sha1: Target entity SHA1
        label: Edge label/type
        metadata: Additional edge attributes
    
    Returns:
        edge_sha1: Content-based hash for the edge
    """
    timestamp = int(time.time() * 1000)
    commit_id = str(uuid.uuid1())
    
    # Prepare edge data
    attr = json.dumps(metadata or {})
    edge_data = {
        "label": label,
        "left": left_sha1,
        "right": right_sha1,
        "attr": attr
    }
    
    # Generate edge SHA1
    edge_sha1 = generate_content_hash(edge_data)
    
    # Store edge
    edge_key = f"edge:head:{commit_id}:{edge_sha1}"
    edge_data["e_sha1"] = edge_sha1
    edge_data["timestamp"] = timestamp
    
    r.hset(edge_key, mapping=edge_data)
    
    print(f"✓ Created edge: {label}")
    print(f"  - From: {left_sha1[:12]}...")
    print(f"  - To: {right_sha1[:12]}...")
    print(f"  - Edge SHA1: {edge_sha1[:12]}...")
    
    return edge_sha1

# Test: Create edges for our test data
# DB → Table edge
db_table_edge = create_entity_edge(
    r, test_db_sha1, test_table_sha1, 
    "contains_table",
    {"schema": "public"}
)

# Table → Column edges
for col_name, (col_sha1, _) in loaded_columns.items():
    create_entity_edge(
        r, test_table_sha1, col_sha1,
        "contains_column",
        {"ordinal_position": list(loaded_columns.keys()).index(col_name) + 1}
    )

In [ ]:
def find_similar_columns(r: redis.Redis, column_sha1: str, 
                         threshold: float = 0.5) -> List[Tuple[str, float]]:
    """
    Find columns with similar value distributions using HLLSet similarity.
    
    Args:
        r: Redis connection
        column_sha1: Column to compare against
        threshold: Minimum Jaccard similarity threshold
    
    Returns:
        List of (column_sha1, similarity_score) tuples
    """
    # Get source column's HLL
    source_meta = r.hgetall(f"meta:column:{column_sha1}")
    if not source_meta:
        return []
    
    source_hll_key = source_meta.get(b"hll_key", b"").decode()
    if not source_hll_key:
        return []
    
    # Search all columns
    similar = []
    
    try:
        results = r.ft("idx:column").search(Query("*").return_fields("column_sha1", "hll_key", "column_name"))
        
        for doc in results.docs:
            other_sha1 = doc.column_sha1 if hasattr(doc, 'column_sha1') else None
            other_hll_key = doc.hll_key if hasattr(doc, 'hll_key') else None
            other_name = doc.column_name if hasattr(doc, 'column_name') else "unknown"
            
            if other_sha1 and other_sha1 != column_sha1 and other_hll_key:
                # Calculate Jaccard similarity using bitmap operations
                # For now, we use a simplified comparison
                # In production, retrieve both HLLs and use hll.match()
                similar.append((other_sha1, other_name, 0.0))  # Placeholder
    except Exception as e:
        print(f"Search error: {e}")
    
    return similar

# Test similarity search
user_id_sha1 = loaded_columns["user_id"][0]
similar_cols = find_similar_columns(r, user_id_sha1)
print(f"\nColumns in index: {similar_cols}")

---
## Part 4: Query and Retrieval Functions

In [ ]:
def get_database_info(r: redis.Redis, db_sha1: str) -> Dict:
    """
    Retrieve complete database information including tables and columns.
    """
    db_data = r.hgetall(f"meta:db:{db_sha1}")
    if not db_data:
        return None
    
    # Decode bytes to strings
    db_info = {k.decode(): v.decode() if isinstance(v, bytes) else v for k, v in db_data.items()}
    
    # Get tables
    tables = []
    table_sha1s = db_info.get("tables", "").split(",")
    for table_sha1 in table_sha1s:
        if table_sha1:
            table_data = r.hgetall(f"meta:table:{table_sha1}")
            if table_data:
                table_info = {k.decode(): v.decode() if isinstance(v, bytes) else v for k, v in table_data.items()}
                
                # Get columns for this table
                columns = []
                column_sha1s = table_info.get("columns", "").split(",")
                for col_sha1 in column_sha1s:
                    if col_sha1:
                        col_data = r.hgetall(f"meta:column:{col_sha1}")
                        if col_data:
                            col_info = {k.decode(): v.decode() if isinstance(v, bytes) else v for k, v in col_data.items()}
                            columns.append(col_info)
                
                table_info["columns_info"] = columns
                tables.append(table_info)
    
    db_info["tables_info"] = tables
    return db_info

# Test: Get complete database info
db_info = get_database_info(r, test_db_sha1)
print(json.dumps(db_info, indent=2, default=str))

In [ ]:
def search_columns_by_name(r: redis.Redis, pattern: str) -> List[Dict]:
    """
    Search for columns by name pattern using Redisearch.
    
    Args:
        r: Redis connection
        pattern: Search pattern (supports wildcards)
    
    Returns:
        List of matching column metadata
    """
    try:
        query = Query(f"@column_name:{pattern}*").return_fields(
            "column_sha1", "column_name", "table_sha1", "data_type", "cardinality"
        )
        results = r.ft("idx:column").search(query)
        
        columns = []
        for doc in results.docs:
            columns.append({
                "column_sha1": getattr(doc, 'column_sha1', None),
                "column_name": getattr(doc, 'column_name', None),
                "table_sha1": getattr(doc, 'table_sha1', None),
                "data_type": getattr(doc, 'data_type', None),
                "cardinality": getattr(doc, 'cardinality', None)
            })
        
        return columns
    except Exception as e:
        print(f"Search error: {e}")
        return []

# Test: Search for columns
print("Searching for columns starting with 'user':")
results = search_columns_by_name(r, "user")
for col in results:
    print(f"  - {col['column_name']}: {col['data_type']} (cardinality: {col['cardinality']})")

In [ ]:
def get_column_statistics(r: redis.Redis, column_sha1: str) -> Dict:
    """
    Get statistics for a column including HLL-based cardinality.
    """
    col_data = r.hgetall(f"meta:column:{column_sha1}")
    if not col_data:
        return None
    
    col_info = {k.decode(): v.decode() if isinstance(v, bytes) else v for k, v in col_data.items()}
    
    # Get table info for context
    table_sha1 = col_info.get("table_sha1")
    table_data = r.hgetall(f"meta:table:{table_sha1}")
    if table_data:
        table_name = table_data.get(b"table_name", b"").decode()
        row_count = int(table_data.get(b"row_count", 0))
        col_info["table_name"] = table_name
        col_info["table_row_count"] = row_count
        
        # Calculate selectivity (unique values / total rows)
        cardinality = int(col_info.get("cardinality", 0))
        if row_count > 0:
            col_info["selectivity"] = cardinality / row_count
        else:
            col_info["selectivity"] = 0
    
    return col_info

# Test: Get column statistics
for col_name, (col_sha1, _) in loaded_columns.items():
    stats = get_column_statistics(r, col_sha1)
    if stats:
        print(f"\n{col_name}:")
        print(f"  Cardinality: {stats.get('cardinality')}")
        print(f"  Data type: {stats.get('data_type')}")
        print(f"  Nullable: {bool(int(stats.get('nullable', 0)))}")

---
## Part 5: Cleanup and Utility Functions

In [ ]:
def list_all_indices(r: redis.Redis) -> List[str]:
    """
    List all Redisearch indices.
    """
    try:
        indices = r.execute_command("FT._LIST")
        return [idx.decode() if isinstance(idx, bytes) else idx for idx in indices]
    except Exception as e:
        print(f"Error listing indices: {e}")
        return []

def get_index_info(r: redis.Redis, index_name: str) -> Dict:
    """
    Get information about a specific Redisearch index.
    """
    try:
        info = r.ft(index_name).info()
        return info
    except Exception as e:
        print(f"Error getting index info: {e}")
        return {}

# List all indices
print("All Redisearch indices:")
for idx in list_all_indices(r):
    print(f"  - {idx}")
    info = get_index_info(r, idx)
    if info:
        num_docs = info.get('num_docs', 'N/A')
        print(f"    Documents: {num_docs}")

In [ ]:
def cleanup_test_data(r: redis.Redis, prefix: str = "meta:") -> int:
    """
    Clean up test data by prefix.
    USE WITH CAUTION!
    
    Args:
        r: Redis connection
        prefix: Key prefix to delete
    
    Returns:
        Number of keys deleted
    """
    keys = r.keys(f"{prefix}*")
    if keys:
        r.delete(*keys)
    print(f"Deleted {len(keys)} keys with prefix '{prefix}'")
    return len(keys)

# Uncomment to clean up test data
# cleanup_test_data(r, "meta:")
# cleanup_test_data(r, "rbs:")
# cleanup_test_data(r, "edge:")

---
## Summary: Functions to Add to meta_redis.py

After prototyping in this notebook, the following functions should be added to `meta_redis.py`:

### Index Creation
- `_create_db_index()` - Database metadata index
- `_create_table_index()` - Table metadata index  
- `_create_column_index()` - Column metadata index with HLL references

### Entity Registration
- `register_database()` - Register a database
- `register_table()` - Register a table and link to database
- `load_column_with_hllset()` - Load column values and create HLLSet
- `load_multiple_columns()` - Batch load multiple columns

### Edge Management
- `create_entity_edge()` - Create edges between entities
- `find_similar_columns()` - Find columns with similar distributions

### Query Functions
- `get_database_info()` - Get full database hierarchy
- `search_columns_by_name()` - Search columns by name pattern
- `get_column_statistics()` - Get column statistics including selectivity

In [ ]:
# Close connection when done
# r.close()
print("Notebook ready for development!")